# 04 - Causal Inference (A/B + DiD)

This notebook runs both randomized and quasi-experimental analyses.


In [1]:
# Cell 0: Setup
from __future__ import annotations

import importlib.util
import json
from datetime import UTC, datetime
from pathlib import Path
import sys
import warnings
import re

required_libs = ['numpy', 'pandas', 'scipy', 'statsmodels', 'joblib', 'sklearn']
missing_libs = [lib for lib in required_libs if importlib.util.find_spec(lib) is None]
if missing_libs:
    raise RuntimeError(
        'Missing required packages for Notebook 04: '
        + ', '.join(missing_libs)
        + '; install with: apps/backend/.venv/bin/python -m pip install -r ml/requirements.txt'
    )

import joblib
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning

ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / 'ml' / 'pipelines' / 'causal_diff_in_diff.py').exists():
        ROOT = p
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CAUSAL_DIR = ROOT / 'ml' / 'data' / 'reports' / 'causal'
AB_DIR = CAUSAL_DIR / 'ab_test'
DID_DIR = CAUSAL_DIR / 'did'
AB_DIR.mkdir(parents=True, exist_ok=True)
DID_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ROOT / 'ml' / 'models' / 'churn_reorder_model.joblib'
METRICS_PATH = ROOT / 'ml' / 'models' / 'churn_reorder_metrics.json'

# Readiness thresholds and business thresholds.
ALPHA = 0.05
SRM_ALPHA = 0.01
MIN_AB_PER_VARIANT = 25
MIN_AB_EVENTS_PER_VARIANT = 5
MIN_BAND_PER_VARIANT = 10
MIN_DID_UNITS = 20
MIN_DID_PERIODS = 4
MIN_DID_ROWS_PER_CELL = 10

WINSORIZE_ENABLED = True
WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99

PRACTICAL_MIN_CONVERSION_LIFT = 0.005
PRACTICAL_MIN_REVENUE_LIFT = 0.0
PRACTICAL_MIN_DID_EFFECT = 0.0

model_artifact = None
model_threshold = None
model_family = None
model_features = []
if MODEL_PATH.exists():
    model_artifact = joblib.load(MODEL_PATH)
    model_threshold = float(model_artifact.get('threshold', 0.5))
    model_family = model_artifact.get('best_model_family')
    model_features = list(model_artifact.get('selected_features', []))
if METRICS_PATH.exists():
    metrics_payload = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
    model_threshold = float(metrics_payload.get('threshold', model_threshold or 0.5))
    model_family = metrics_payload.get('best_model_family', model_family)

print('ROOT:', ROOT)
print('Model artifact found:', MODEL_PATH.exists())
print('Model family:', model_family)
print('Model threshold:', model_threshold)
print('Model feature count:', len(model_features))
print('Guardrails:', {
    'ALPHA': ALPHA,
    'SRM_ALPHA': SRM_ALPHA,
    'MIN_AB_PER_VARIANT': MIN_AB_PER_VARIANT,
    'MIN_AB_EVENTS_PER_VARIANT': MIN_AB_EVENTS_PER_VARIANT,
    'MIN_BAND_PER_VARIANT': MIN_BAND_PER_VARIANT,
    'MIN_DID_UNITS': MIN_DID_UNITS,
    'MIN_DID_PERIODS': MIN_DID_PERIODS,
    'MIN_DID_ROWS_PER_CELL': MIN_DID_ROWS_PER_CELL,
})




ROOT: /Users/deliorincon/Desktop/Sliceiq
Model artifact found: True
Model family: extra_trees
Model threshold: 0.31247084061008906
Model feature count: 24
Guardrails: {'ALPHA': 0.05, 'SRM_ALPHA': 0.01, 'MIN_AB_PER_VARIANT': 25, 'MIN_AB_EVENTS_PER_VARIANT': 5, 'MIN_BAND_PER_VARIANT': 10, 'MIN_DID_UNITS': 20, 'MIN_DID_PERIODS': 4, 'MIN_DID_ROWS_PER_CELL': 10}


In [2]:
# Cell 1: A/B load + SRM + covariate balance + risk segmentation
ab_path = ROOT / 'ml' / 'data' / 'experiments' / 'ab_checkout_template.csv'
assert ab_path.exists(), 'Missing A/B dataset template. Expected ml/data/experiments/ab_checkout_template.csv'

ab = pd.read_csv(ab_path)
ab['variant'] = ab['variant'].astype(str)
ab['converted'] = pd.to_numeric(ab.get('converted', 0), errors='coerce').fillna(0).astype(int)
ab['revenue_30d'] = pd.to_numeric(ab.get('revenue_30d', 0.0), errors='coerce').fillna(0.0)
ab['pre_revenue_30d'] = pd.to_numeric(ab.get('pre_revenue_30d', 0.0), errors='coerce').fillna(0.0)
if 'pre_orders' in ab.columns:
    ab['pre_orders'] = pd.to_numeric(ab['pre_orders'], errors='coerce').fillna(0.0)

variants = sorted(ab['variant'].dropna().unique().tolist())
if len(variants) != 2:
    raise RuntimeError(f'Expected exactly 2 variants, found {len(variants)}: {variants}')
control, treatment = variants[0], variants[1]
ab['is_treatment'] = (ab['variant'] == treatment).astype(int)


def _rank01(s: pd.Series) -> pd.Series:
    vals = pd.to_numeric(s, errors='coerce')
    if vals.notna().sum() == 0:
        return pd.Series(np.repeat(0.5, len(vals)), index=vals.index)
    ranked = vals.rank(method='average', pct=True)
    return ranked.fillna(ranked.median()).clip(0.0, 1.0)


def _make_risk_band(score: pd.Series, q: int = 3) -> pd.Series:
    n_unique = int(score.nunique(dropna=True))
    if n_unique < 2:
        return pd.Series(np.repeat('mid', len(score)), index=score.index)
    bins = min(q, n_unique)
    labels = ['low', 'mid', 'high'][:bins]
    ranked = score.rank(method='first')
    return pd.qcut(ranked, q=bins, labels=labels, duplicates='drop').astype(str)


def _safe_smd(frame: pd.DataFrame, col: str, control_label: str, treatment_label: str) -> float:
    c = pd.to_numeric(frame.loc[frame['variant'] == control_label, col], errors='coerce').dropna()
    t = pd.to_numeric(frame.loc[frame['variant'] == treatment_label, col], errors='coerce').dropna()
    if len(c) < 2 or len(t) < 2:
        return np.nan
    c_var = c.var(ddof=1)
    t_var = t.var(ddof=1)
    pooled = np.sqrt((c_var + t_var) / 2.0)
    if pooled <= 0:
        return 0.0
    return float((t.mean() - c.mean()) / pooled)


if model_artifact is not None and model_features and set(model_features).issubset(ab.columns):
    X_ab = ab[model_features].apply(pd.to_numeric, errors='coerce')
    ab['risk_score'] = model_artifact['model'].predict_proba(X_ab)[:, 1]
    risk_source = 'model_score'
else:
    ab['risk_score'] = 1.0 - _rank01(ab['pre_revenue_30d'])
    risk_source = 'pre_revenue_proxy'

ab['risk_band'] = _make_risk_band(ab['risk_score'], q=3)

# Sample ratio mismatch (SRM) test vs expected 50/50 split.
sample_sizes = ab['variant'].value_counts().to_dict()
n_control = int(sample_sizes.get(control, 0))
n_treatment = int(sample_sizes.get(treatment, 0))
n_total = n_control + n_treatment
expected = np.array([n_total / 2.0, n_total / 2.0]) if n_total > 0 else np.array([0.0, 0.0])
observed = np.array([n_control, n_treatment], dtype=float)
if n_total > 0 and expected.min() > 0:
    srm_chi2 = float(((observed - expected) ** 2 / expected).sum())
    srm_p_value = float(1.0 - stats.chi2.cdf(srm_chi2, df=1))
else:
    srm_chi2 = np.nan
    srm_p_value = np.nan
srm_flag = pd.notna(srm_p_value) and srm_p_value < SRM_ALPHA

balance = (
    ab.groupby(['variant', 'risk_band'], as_index=False)
    .agg(users=('converted', 'size'), pre_rev_mean=('pre_revenue_30d', 'mean'))
    .sort_values(['risk_band', 'variant'])
)

band_variant_counts = ab.groupby(['risk_band', 'variant']).size().unstack(fill_value=0)
for v in [control, treatment]:
    if v not in band_variant_counts.columns:
        band_variant_counts[v] = 0
band_variant_counts = band_variant_counts[[control, treatment]]

covariate_cols = [c for c in ['pre_revenue_30d', 'pre_orders', 'risk_score'] if c in ab.columns]
balance_smd_rows = []
for col in covariate_cols:
    balance_smd_rows.append({'covariate': col, 'smd_treat_minus_control': _safe_smd(ab, col, control, treatment)})
balance_smd = pd.DataFrame(balance_smd_rows)

ratio = n_treatment / max(1, n_control)
sample_ratio_mismatch = abs(ratio - 1.0)

ab_binary_skip_reasons = []
min_variant_n = min(n_control, n_treatment)
if min_variant_n < MIN_AB_PER_VARIANT:
    ab_binary_skip_reasons.append(f'min variant size {min_variant_n} < MIN_AB_PER_VARIANT {MIN_AB_PER_VARIANT}')

conv_stats = (
    ab.groupby('variant')['converted']
    .agg(success='sum', n='count')
    .reindex([control, treatment])
    .fillna(0)
)
for v in [control, treatment]:
    success = int(conv_stats.loc[v, 'success'])
    total = int(conv_stats.loc[v, 'n'])
    failure = total - success
    if success < MIN_AB_EVENTS_PER_VARIANT or failure < MIN_AB_EVENTS_PER_VARIANT:
        ab_binary_skip_reasons.append(
            f'variant={v} has success={success}, failure={failure}, needs >= {MIN_AB_EVENTS_PER_VARIANT} each'
        )
if srm_flag:
    ab_binary_skip_reasons.append(f'SRM failed (p={srm_p_value:.6g} < {SRM_ALPHA})')

ab_binary_ready = len(ab_binary_skip_reasons) == 0
ab_continuous_ready = (min_variant_n >= MIN_AB_PER_VARIANT) and (not srm_flag)

ab_heterogeneity_skip_reasons = []
bands_with_both = int(((band_variant_counts[control] >= MIN_BAND_PER_VARIANT) & (band_variant_counts[treatment] >= MIN_BAND_PER_VARIANT)).sum())
if bands_with_both < 2:
    ab_heterogeneity_skip_reasons.append(
        f'fewer than 2 risk bands with >= {MIN_BAND_PER_VARIANT} users per variant'
    )
if ab['risk_score'].nunique(dropna=True) < 5:
    ab_heterogeneity_skip_reasons.append('risk_score has too few unique values (<5)')
if not ab_binary_ready:
    ab_heterogeneity_skip_reasons.append('binary inference is not ready')
ab_heterogeneity_ready = len(ab_heterogeneity_skip_reasons) == 0

print('control:', control, '| treatment:', treatment)
print('risk_source:', risk_source)
print('sample_sizes:', sample_sizes)
print('sample_ratio_mismatch:', round(sample_ratio_mismatch, 4))
print('srm_chi2:', srm_chi2, '| srm_p_value:', srm_p_value, '| srm_flag:', srm_flag)
print('ab_binary_ready:', ab_binary_ready)
print('ab_continuous_ready:', ab_continuous_ready)
print('ab_heterogeneity_ready:', ab_heterogeneity_ready)
if ab_binary_skip_reasons:
    print('ab_binary_skip_reasons:', ab_binary_skip_reasons)
if ab_heterogeneity_skip_reasons:
    print('ab_heterogeneity_skip_reasons:', ab_heterogeneity_skip_reasons)

display(balance)
display(band_variant_counts.reset_index())
display(balance_smd)



control: control | treatment: treatment
risk_source: model_score
sample_sizes: {'control': 1200, 'treatment': 1200}
sample_ratio_mismatch: 0.0
srm_chi2: 0.0 | srm_p_value: 1.0 | srm_flag: False
ab_binary_ready: True
ab_continuous_ready: True
ab_heterogeneity_ready: True


,variant,risk_band,users,pre_rev_mean
0,control,high,403,108.927556
3,treatment,high,397,105.733913
1,control,low,404,10.100099
4,treatment,low,396,11.455253
2,control,mid,393,44.666336
5,treatment,mid,407,42.420725


variant,risk_band,control,treatment
0,high,403,397
1,low,404,396
2,mid,393,407


,covariate,smd_treat_minus_control
0,pre_revenue_30d,-0.024929
1,pre_orders,-0.031024
2,risk_score,0.000033


In [3]:
# Cell 2: A/B binary outcome (overall + risk heterogeneity + BH correction)
g = (
    ab.groupby('variant')['converted']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'successes', 'count': 'n', 'mean': 'rate'})
    .reindex([control, treatment])
)

rate_diff = np.nan
rate_diff_ci_low = np.nan
rate_diff_ci_high = np.nan
relative_lift = np.nan
z_stat = np.nan
p_value = np.nan
binary_note = 'computed'
if ab_binary_ready:
    count = np.array([g.loc[treatment, 'successes'], g.loc[control, 'successes']], dtype=float)
    nobs = np.array([g.loc[treatment, 'n'], g.loc[control, 'n']], dtype=float)
    z_stat, p_value = proportions_ztest(count, nobs)

    p_t = float(g.loc[treatment, 'rate'])
    p_c = float(g.loc[control, 'rate'])
    n_t = float(g.loc[treatment, 'n'])
    n_c = float(g.loc[control, 'n'])
    rate_diff = p_t - p_c
    relative_lift = (p_t / p_c - 1.0) if p_c > 0 else np.nan
    se = np.sqrt((p_t * (1 - p_t) / max(1.0, n_t)) + (p_c * (1 - p_c) / max(1.0, n_c)))
    zcrit = stats.norm.ppf(1 - ALPHA / 2)
    rate_diff_ci_low = rate_diff - zcrit * se
    rate_diff_ci_high = rate_diff + zcrit * se
else:
    binary_note = 'skipped_due_to_insufficient_data'

binary_practical_ok = pd.notna(rate_diff) and (float(rate_diff) >= PRACTICAL_MIN_CONVERSION_LIFT)

risk_binary = (
    ab.groupby(['risk_band', 'variant'])['converted']
    .mean()
    .reset_index()
    .pivot(index='risk_band', columns='variant', values='converted')
)
for v in [control, treatment]:
    if v not in risk_binary.columns:
        risk_binary[v] = np.nan
risk_binary['rate_diff_treat_minus_control'] = risk_binary[treatment] - risk_binary[control]
risk_binary = risk_binary.reset_index()

risk_band_binary_tests_rows = []
for band, chunk in ab.groupby('risk_band'):
    n_c = int((chunk['variant'] == control).sum())
    n_t = int((chunk['variant'] == treatment).sum())
    if n_c < MIN_BAND_PER_VARIANT or n_t < MIN_BAND_PER_VARIANT:
        continue

    sub_g = chunk.groupby('variant')['converted'].agg(['sum', 'count', 'mean']).reindex([control, treatment]).fillna(0)
    ct = np.array([sub_g.loc[treatment, 'sum'], sub_g.loc[control, 'sum']], dtype=float)
    nt = np.array([sub_g.loc[treatment, 'count'], sub_g.loc[control, 'count']], dtype=float)
    try:
        z_b, p_b = proportions_ztest(ct, nt)
    except Exception:
        z_b, p_b = np.nan, np.nan

    risk_band_binary_tests_rows.append(
        {
            'risk_band': str(band),
            'n_control': n_c,
            'n_treatment': n_t,
            'control_rate': float(sub_g.loc[control, 'mean']),
            'treatment_rate': float(sub_g.loc[treatment, 'mean']),
            'rate_diff': float(sub_g.loc[treatment, 'mean'] - sub_g.loc[control, 'mean']),
            'z_stat': None if pd.isna(z_b) else float(z_b),
            'p_value': None if pd.isna(p_b) else float(p_b),
        }
    )

risk_band_binary_tests = pd.DataFrame(risk_band_binary_tests_rows)
if not risk_band_binary_tests.empty and risk_band_binary_tests['p_value'].notna().sum() > 0:
    valid = risk_band_binary_tests['p_value'].notna()
    reject, qvals, _, _ = multipletests(risk_band_binary_tests.loc[valid, 'p_value'], alpha=ALPHA, method='fdr_bh')
    risk_band_binary_tests.loc[valid, 'q_value_bh'] = qvals
    risk_band_binary_tests.loc[valid, 'significant_fdr_bh'] = reject.astype(bool)

binary_interaction_coef = np.nan
binary_interaction_p = np.nan
binary_interaction_note = 'computed'
if ab_heterogeneity_ready:
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('error', category=PerfectSeparationWarning)
            glm = smf.glm(
                'converted ~ is_treatment + risk_score + is_treatment:risk_score',
                data=ab,
                family=sm.families.Binomial(),
            ).fit()
        binary_interaction_coef = float(glm.params.get('is_treatment:risk_score', np.nan))
        binary_interaction_p = float(glm.pvalues.get('is_treatment:risk_score', np.nan))
    except Exception as exc:
        binary_interaction_note = f'skipped_due_to_model_instability: {exc}'
else:
    binary_interaction_note = 'skipped_due_to_insufficient_data'

print('Overall conversion rate diff:', rate_diff)
print('Overall conversion p-value:', p_value)
print('binary_practical_ok:', binary_practical_ok)
print('binary_note:', binary_note)
print('Binary heterogeneity interaction coef:', binary_interaction_coef, '| p-value:', binary_interaction_p)
print('binary_interaction_note:', binary_interaction_note)

display(risk_binary)
if not risk_band_binary_tests.empty:
    display(risk_band_binary_tests)




Overall conversion rate diff: 0.13666666666666666
Overall conversion p-value: 9.857681621527965e-12
binary_practical_ok: True
binary_note: computed
Binary heterogeneity interaction coef: 0.5067491315740117 | p-value: 0.020587817981158586
binary_interaction_note: computed


variant,risk_band,control,treatment,rate_diff_treat_minus_control
0,high,0.146402,0.292191,0.145789
1,low,0.549505,0.651515,0.102010
2,mid,0.325700,0.488943,0.163244


,risk_band,n_control,n_treatment,control_rate,treatment_rate,rate_diff,z_stat,p_value,q_value_bh,significant_fdr_bh
0,high,403,397,0.146402,0.292191,0.145789,4.987235,6.124953e-07,0.000002,True
1,low,404,396,0.549505,0.651515,0.102010,2.944634,3.233373e-03,0.003233,True
2,mid,393,407,0.325700,0.488943,0.163244,4.695378,2.661140e-06,0.000004,True


In [4]:
# Cell 3: A/B continuous outcomes (winsorized Welch + CUPED + heterogeneity + BH)
metric_col = 'revenue_30d'
work = ab[['variant', 'is_treatment', 'risk_score', 'risk_band', 'pre_revenue_30d', metric_col]].copy()

if WINSORIZE_ENABLED and work[metric_col].notna().sum() >= 10:
    lo, hi = work[metric_col].quantile([WINSOR_LOWER_Q, WINSOR_UPPER_Q])
    work['revenue_analysis'] = work[metric_col].clip(lower=lo, upper=hi)
    winsor_note = f'winsorized_{WINSOR_LOWER_Q:.2f}_{WINSOR_UPPER_Q:.2f}'
else:
    work['revenue_analysis'] = work[metric_col]
    winsor_note = 'raw_revenue'

control_vals = work.loc[work['variant'] == control, 'revenue_analysis'].to_numpy(dtype=float)
treat_vals = work.loc[work['variant'] == treatment, 'revenue_analysis'].to_numpy(dtype=float)

welch_t = np.nan
welch_p = np.nan
mu_diff = np.nan
mu_ci_low = np.nan
mu_ci_high = np.nan
cohen_d = np.nan
continuous_note = 'computed'

if ab_continuous_ready:
    welch_t, welch_p = stats.ttest_ind(treat_vals, control_vals, equal_var=False)
    mu_diff = float(np.mean(treat_vals) - np.mean(control_vals))

    n_t = len(treat_vals)
    n_c = len(control_vals)
    s2_t = np.var(treat_vals, ddof=1) if n_t > 1 else np.nan
    s2_c = np.var(control_vals, ddof=1) if n_c > 1 else np.nan
    se = np.sqrt(s2_t / n_t + s2_c / n_c) if n_t > 1 and n_c > 1 else np.nan
    if pd.notna(se) and se > 0:
        df_num = (s2_t / n_t + s2_c / n_c) ** 2
        df_den = ((s2_t / n_t) ** 2) / max(1, n_t - 1) + ((s2_c / n_c) ** 2) / max(1, n_c - 1)
        df_welch = df_num / df_den if df_den > 0 else np.nan
        tcrit = stats.t.ppf(1 - ALPHA / 2, df_welch) if pd.notna(df_welch) else np.nan
        if pd.notna(tcrit):
            mu_ci_low = mu_diff - tcrit * se
            mu_ci_high = mu_diff + tcrit * se

    if n_t > 1 and n_c > 1:
        pooled_sd = np.sqrt(
            (((n_t - 1) * s2_t) + ((n_c - 1) * s2_c)) / max(1, n_t + n_c - 2)
        )
        cohen_d = (mu_diff / pooled_sd) if pooled_sd > 0 else 0.0
else:
    continuous_note = 'skipped_due_to_insufficient_data'

revenue_practical_ok = pd.notna(mu_diff) and (float(mu_diff) >= PRACTICAL_MIN_REVENUE_LIFT)

# CUPED on revenue_analysis.
theta = np.nan
cuped_t = np.nan
cuped_p = np.nan
cuped_diff = np.nan
cuped_var_reduction = np.nan
cuped_note = 'computed'

if ab_continuous_ready:
    x = work['pre_revenue_30d'].to_numpy(dtype=float)
    y = work['revenue_analysis'].to_numpy(dtype=float)
    y_var = float(np.var(y, ddof=1)) if len(y) > 1 else np.nan
    theta = 0.0 if np.var(x, ddof=1) <= 0 else float(np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1))
    work['cuped_adjusted'] = y - theta * (x - x.mean())

    control_adj = work.loc[work['variant'] == control, 'cuped_adjusted'].to_numpy(dtype=float)
    treat_adj = work.loc[work['variant'] == treatment, 'cuped_adjusted'].to_numpy(dtype=float)
    cuped_t, cuped_p = stats.ttest_ind(treat_adj, control_adj, equal_var=False)
    cuped_diff = float(np.mean(treat_adj) - np.mean(control_adj))

    adj_var = float(np.var(work['cuped_adjusted'].to_numpy(dtype=float), ddof=1)) if len(work) > 1 else np.nan
    if pd.notna(y_var) and y_var > 0 and pd.notna(adj_var):
        cuped_var_reduction = float(1.0 - adj_var / y_var)
else:
    work['cuped_adjusted'] = np.nan
    cuped_note = 'skipped_due_to_insufficient_data'

risk_band_cont_tests_rows = []
for band, chunk in work.groupby('risk_band'):
    c_vals = chunk.loc[chunk['variant'] == control, 'revenue_analysis'].to_numpy(dtype=float)
    t_vals = chunk.loc[chunk['variant'] == treatment, 'revenue_analysis'].to_numpy(dtype=float)
    if len(c_vals) < MIN_BAND_PER_VARIANT or len(t_vals) < MIN_BAND_PER_VARIANT:
        continue
    t_b, p_b = stats.ttest_ind(t_vals, c_vals, equal_var=False)
    risk_band_cont_tests_rows.append(
        {
            'risk_band': str(band),
            'n_control': int(len(c_vals)),
            'n_treatment': int(len(t_vals)),
            'control_mean': float(np.mean(c_vals)),
            'treatment_mean': float(np.mean(t_vals)),
            'difference': float(np.mean(t_vals) - np.mean(c_vals)),
            'welch_t': float(t_b),
            'p_value': float(p_b),
        }
    )

risk_band_cont_tests = pd.DataFrame(risk_band_cont_tests_rows)
if not risk_band_cont_tests.empty and risk_band_cont_tests['p_value'].notna().sum() > 0:
    valid = risk_band_cont_tests['p_value'].notna()
    reject, qvals, _, _ = multipletests(risk_band_cont_tests.loc[valid, 'p_value'], alpha=ALPHA, method='fdr_bh')
    risk_band_cont_tests.loc[valid, 'q_value_bh'] = qvals
    risk_band_cont_tests.loc[valid, 'significant_fdr_bh'] = reject.astype(bool)

revenue_interaction_coef = np.nan
revenue_interaction_p = np.nan
revenue_interaction_note = 'computed'
if ab_heterogeneity_ready and ab_continuous_ready:
    try:
        reg = smf.ols(
            'revenue_analysis ~ is_treatment + risk_score + is_treatment:risk_score + pre_revenue_30d',
            data=work,
        ).fit(cov_type='HC3')
        revenue_interaction_coef = float(reg.params.get('is_treatment:risk_score', np.nan))
        revenue_interaction_p = float(reg.pvalues.get('is_treatment:risk_score', np.nan))
    except Exception as exc:
        revenue_interaction_note = f'skipped_due_to_model_instability: {exc}'
else:
    revenue_interaction_note = 'skipped_due_to_insufficient_data'

print('Revenue metric mode:', winsor_note)
print('Welch revenue diff:', mu_diff, '| p:', welch_p, '| note:', continuous_note)
print('Cohen d:', cohen_d, '| CI:', (mu_ci_low, mu_ci_high), '| revenue_practical_ok:', revenue_practical_ok)
print('CUPED diff:', cuped_diff, '| p:', cuped_p, '| theta:', theta, '| var_reduction:', cuped_var_reduction)
print('Revenue heterogeneity interaction coef:', revenue_interaction_coef, '| p:', revenue_interaction_p)
print('revenue_interaction_note:', revenue_interaction_note)
if not risk_band_cont_tests.empty:
    display(risk_band_cont_tests)



Revenue metric mode: winsorized_0.01_0.99
Welch revenue diff: 24.585288333333338 | p: 1.3032580911521324e-14 | note: computed
Cohen d: 0.3166016351343911 | CI: (np.float64(18.368619782131844), np.float64(30.80195688453483)) | revenue_practical_ok: True
CUPED diff: 24.334055040235526 | p: 1.471881538223187e-14 | theta: -0.17185819761151722 | var_reduction: 0.016433091395768318
Revenue heterogeneity interaction coef: -0.5394128027272969 | p: 0.9408465067120342
revenue_interaction_note: computed


,risk_band,n_control,n_treatment,control_mean,treatment_mean,difference,welch_t,p_value,q_value_bh,significant_fdr_bh
0,high,403,397,21.832233,46.689547,24.857313,5.335280,1.277830e-07,3.833491e-07,True
1,low,404,396,58.494437,84.883879,26.389442,4.453142,9.686737e-06,1.453011e-05,True
2,mid,393,407,45.791913,68.252934,22.461021,4.078398,4.989805e-05,4.989805e-05,True


In [5]:
# Cell 4: Save A/B report
ab_results = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'control_label': control,
    'treatment_label': treatment,
    'risk_source': risk_source,
    'sample_ratio_mismatch': float(sample_ratio_mismatch),
    'srm': {
        'chi2': None if pd.isna(srm_chi2) else float(srm_chi2),
        'p_value': None if pd.isna(srm_p_value) else float(srm_p_value),
        'flag': bool(srm_flag),
        'alpha': SRM_ALPHA,
    },
    'readiness': {
        'ab_binary_ready': bool(ab_binary_ready),
        'ab_continuous_ready': bool(ab_continuous_ready),
        'ab_heterogeneity_ready': bool(ab_heterogeneity_ready),
        'ab_binary_skip_reasons': ab_binary_skip_reasons,
        'ab_heterogeneity_skip_reasons': ab_heterogeneity_skip_reasons,
    },
    'covariate_balance_smd': balance_smd.to_dict(orient='records'),
    'binary_results': {
        'control_rate': float(g.loc[control, 'rate']) if pd.notna(g.loc[control, 'rate']) else None,
        'treatment_rate': float(g.loc[treatment, 'rate']) if pd.notna(g.loc[treatment, 'rate']) else None,
        'rate_diff': None if pd.isna(rate_diff) else float(rate_diff),
        'rate_diff_ci_low': None if pd.isna(rate_diff_ci_low) else float(rate_diff_ci_low),
        'rate_diff_ci_high': None if pd.isna(rate_diff_ci_high) else float(rate_diff_ci_high),
        'relative_lift': None if pd.isna(relative_lift) else float(relative_lift),
        'z_stat': None if pd.isna(z_stat) else float(z_stat),
        'p_value': None if pd.isna(p_value) else float(p_value),
        'practical_min_lift': PRACTICAL_MIN_CONVERSION_LIFT,
        'practical_ok': bool(binary_practical_ok),
        'note': binary_note,
        'risk_interaction_coef': None if pd.isna(binary_interaction_coef) else float(binary_interaction_coef),
        'risk_interaction_p_value': None if pd.isna(binary_interaction_p) else float(binary_interaction_p),
        'risk_interaction_note': binary_interaction_note,
    },
    'continuous_results': {
        'metric': 'revenue_analysis',
        'metric_mode': winsor_note,
        'difference': None if pd.isna(mu_diff) else float(mu_diff),
        'difference_ci_low': None if pd.isna(mu_ci_low) else float(mu_ci_low),
        'difference_ci_high': None if pd.isna(mu_ci_high) else float(mu_ci_high),
        'cohen_d': None if pd.isna(cohen_d) else float(cohen_d),
        'welch_t': None if pd.isna(welch_t) else float(welch_t),
        'welch_p_value': None if pd.isna(welch_p) else float(welch_p),
        'practical_min_lift': PRACTICAL_MIN_REVENUE_LIFT,
        'practical_ok': bool(revenue_practical_ok),
        'note': continuous_note,
        'risk_interaction_coef': None if pd.isna(revenue_interaction_coef) else float(revenue_interaction_coef),
        'risk_interaction_p_value': None if pd.isna(revenue_interaction_p) else float(revenue_interaction_p),
        'risk_interaction_note': revenue_interaction_note,
    },
    'cuped_results': {
        'difference': None if pd.isna(cuped_diff) else float(cuped_diff),
        'welch_t': None if pd.isna(cuped_t) else float(cuped_t),
        'welch_p_value': None if pd.isna(cuped_p) else float(cuped_p),
        'theta': None if pd.isna(theta) else float(theta),
        'variance_reduction': None if pd.isna(cuped_var_reduction) else float(cuped_var_reduction),
        'note': cuped_note,
    },
    'risk_band_binary_uplift': risk_binary.to_dict(orient='records'),
    'risk_band_binary_tests': risk_band_binary_tests.to_dict(orient='records') if 'risk_band_binary_tests' in globals() else [],
    'risk_band_continuous_tests': risk_band_cont_tests.to_dict(orient='records') if 'risk_band_cont_tests' in globals() else [],
}

ab_out = AB_DIR / 'ab_test_results.json'
ab_out.write_text(json.dumps(ab_results, indent=2, default=str), encoding='utf-8')

if 'risk_band_binary_tests' in globals() and isinstance(risk_band_binary_tests, pd.DataFrame):
    risk_band_binary_tests.to_csv(AB_DIR / 'ab_risk_band_binary_tests.csv', index=False)
if 'risk_band_cont_tests' in globals() and isinstance(risk_band_cont_tests, pd.DataFrame):
    risk_band_cont_tests.to_csv(AB_DIR / 'ab_risk_band_continuous_tests.csv', index=False)

print('Saved A/B results:', ab_out)




Saved A/B results: /Users/deliorincon/Desktop/Sliceiq/ml/data/reports/causal/ab_test/ab_test_results.json


In [6]:
# Cell 5: DiD load + readiness gates + event-time prep
did_path = ROOT / 'ml' / 'data' / 'experiments' / 'did_panel_template.csv'
assert did_path.exists(), 'Missing DiD dataset template. Expected ml/data/experiments/did_panel_template.csv'

did = pd.read_csv(did_path)
did['outcome'] = pd.to_numeric(did['outcome'], errors='coerce')
did['treated'] = pd.to_numeric(did['treated'], errors='coerce').fillna(0).astype(int)
did['post'] = pd.to_numeric(did['post'], errors='coerce').fillna(0).astype(int)
did['period'] = pd.to_datetime(did['period'], errors='coerce')
did = did.dropna(subset=['outcome', 'period']).copy()

if 'user_id' not in did.columns:
    did['user_id'] = np.arange(len(did)).astype(str)

did['treated_post'] = did['treated'] * did['post']
did['period_str'] = did['period'].dt.strftime('%Y-%m')

if 'pre_orders' in did.columns:
    did['pre_orders'] = pd.to_numeric(did['pre_orders'], errors='coerce').fillna(0.0)
    did['risk_score'] = 1.0 - _rank01(did.groupby('user_id')['pre_orders'].transform('max'))
else:
    did['risk_score'] = _rank01(did.groupby('user_id')['outcome'].transform('mean'))

did['risk_band'] = _make_risk_band(did['risk_score'], q=3)

# Build relative period index for event-study.
period_order = {k: i for i, k in enumerate(sorted(did['period_str'].dropna().unique().tolist()))}
did['period_idx'] = did['period_str'].map(period_order).astype(int)
if (did['post'] == 1).any():
    first_post_idx = int(did.loc[did['post'] == 1, 'period_idx'].min())
else:
    first_post_idx = int(did['period_idx'].max())
did['rel_period'] = did['period_idx'] - first_post_idx
did['rel_period_cat'] = did['rel_period'].astype(int).astype(str)

did_cell_counts = did.groupby(['treated', 'post']).size().to_dict()
required_cells = {(0, 0), (0, 1), (1, 0), (1, 1)}
missing_cells = sorted(list(required_cells - set(did_cell_counts.keys())))
min_cell_rows = min([did_cell_counts.get(k, 0) for k in required_cells]) if did_cell_counts else 0

did_skip_reasons = []
if did['user_id'].nunique() < MIN_DID_UNITS:
    did_skip_reasons.append(f"unique units {did['user_id'].nunique()} < MIN_DID_UNITS {MIN_DID_UNITS}")
if did['period_str'].nunique() < MIN_DID_PERIODS:
    did_skip_reasons.append(f"periods {did['period_str'].nunique()} < MIN_DID_PERIODS {MIN_DID_PERIODS}")
if missing_cells:
    did_skip_reasons.append(f'missing treated/post cells: {missing_cells}')
if min_cell_rows < MIN_DID_ROWS_PER_CELL:
    did_skip_reasons.append(
        f'min rows per treated/post cell {min_cell_rows} < MIN_DID_ROWS_PER_CELL {MIN_DID_ROWS_PER_CELL}'
    )

did_ready_for_manual = len(missing_cells) == 0
did_ready_for_regression = len(did_skip_reasons) == 0

# Event-study readiness.
pre_periods = sorted(did.loc[did['rel_period'] < 0, 'rel_period'].unique().tolist())
post_periods = sorted(did.loc[did['rel_period'] >= 0, 'rel_period'].unique().tolist())
event_study_ready = did_ready_for_regression and (-1 in did['rel_period'].unique()) and (len(pre_periods) >= 2) and (len(post_periods) >= 2)

display(did.head())
print('DiD rows:', len(did))
print('Unique units:', did['user_id'].nunique())
print('Periods:', sorted(did['period_str'].dropna().unique().tolist()))
print('treated/post cell counts:', did_cell_counts)
print('did_ready_for_manual:', did_ready_for_manual)
print('did_ready_for_regression:', did_ready_for_regression)
print('event_study_ready:', event_study_ready)
if did_skip_reasons:
    print('did_skip_reasons:', did_skip_reasons)



,user_id,period,outcome,treated,post,pre_orders,treated_post,period_str,risk_score,risk_band,period_idx,rel_period,rel_period_cat
0,1,2026-02-05,1.0488,0,1,1.0,0,2026-02,0.744780,mid,3,1,1
1,2,2025-11-20,2.1356,1,0,2.0,0,2025-11,0.197216,low,0,-2,-2
2,2,2025-11-27,1.8829,1,0,2.0,0,2025-11,0.197216,low,0,-2,-2
3,2,2025-12-04,3.8183,1,0,2.0,0,2025-12,0.197216,low,1,-1,-1
4,2,2025-12-11,3.6016,1,0,4.0,0,2025-12,0.197216,low,1,-1,-1


DiD rows: 862
Unique units: 135
Periods: ['2025-11', '2025-12', '2026-01', '2026-02']
treated/post cell counts: {(0, 0): 205, (0, 1): 219, (1, 0): 226, (1, 1): 212}
did_ready_for_manual: True
did_ready_for_regression: True
event_study_ready: True


In [7]:
# Cell 6: Manual DiD (overall + risk-band)

def manual_did_from_frame(frame: pd.DataFrame) -> tuple[dict[str, float], pd.DataFrame]:
    means = frame.groupby(['treated', 'post'], as_index=False)['outcome'].mean()
    lookup = {(int(r.treated), int(r.post)): float(r.outcome) for _, r in means.iterrows()}
    control_pre = lookup.get((0, 0), np.nan)
    control_post = lookup.get((0, 1), np.nan)
    treat_pre = lookup.get((1, 0), np.nan)
    treat_post = lookup.get((1, 1), np.nan)
    did_val = (treat_post - treat_pre) - (control_post - control_pre)
    return {
        'control_pre': control_pre,
        'control_post': control_post,
        'treatment_pre': treat_pre,
        'treatment_post': treat_post,
        'did_manual': did_val,
    }, means

if did_ready_for_manual:
    manual_overall, means_overall = manual_did_from_frame(did)
    display(means_overall)
    print('Manual DiD overall:', manual_overall['did_manual'])
else:
    means_overall = did.groupby(['treated', 'post'], as_index=False)['outcome'].mean()
    manual_overall = {
        'control_pre': np.nan,
        'control_post': np.nan,
        'treatment_pre': np.nan,
        'treatment_post': np.nan,
        'did_manual': np.nan,
    }
    display(means_overall)
    print('Manual DiD skipped: missing at least one treated/post cell.')

risk_did_rows = []
for band, chunk in did.groupby('risk_band'):
    cells_here = set(map(tuple, chunk[['treated', 'post']].drop_duplicates().to_records(index=False)))
    if not required_cells.issubset(cells_here):
        continue
    m, _ = manual_did_from_frame(chunk)
    risk_did_rows.append(
        {
            'risk_band': str(band),
            'did_manual': float(m['did_manual']),
            'control_pre': float(m['control_pre']),
            'control_post': float(m['control_post']),
            'treatment_pre': float(m['treatment_pre']),
            'treatment_post': float(m['treatment_post']),
            'rows': int(len(chunk)),
        }
    )

risk_did_table = pd.DataFrame(risk_did_rows)
if not risk_did_table.empty:
    risk_did_table = risk_did_table.sort_values('risk_band').reset_index(drop=True)
    display(risk_did_table)
else:
    risk_did_table = pd.DataFrame(
        columns=['risk_band', 'did_manual', 'control_pre', 'control_post', 'treatment_pre', 'treatment_post', 'rows']
    )
    print('No risk-band DiD estimates (insufficient support per risk band).')



,treated,post,outcome
0,0,0,2.379264
1,0,1,2.331871
2,1,0,1.800843
3,1,1,2.464838


Manual DiD overall: 0.7113874992005147


,risk_band,did_manual,control_pre,control_post,treatment_pre,treatment_post,rows
0,high,0.503177,0.680056,0.651271,0.366975,0.841367,287
1,low,1.546790,4.314281,5.263148,3.166357,5.662014,288
2,mid,0.491126,1.581001,1.507149,1.454920,1.872194,287


In [8]:
# Cell 7: Regression DiD (basic + TWFE clustered) + event-study + placebo
cov_terms = ''
if 'pre_orders' in did.columns:
    cov_terms = ' + pre_orders'

basic_formula = f'outcome ~ treated + post + treated:post{cov_terms}'
if did['risk_score'].nunique(dropna=True) > 1:
    basic_formula += ' + risk_score + treated:post:risk_score'

twfe_formula = f'outcome ~ treated:post{cov_terms} + C(user_id) + C(period_str)'
if did['risk_score'].nunique(dropna=True) > 1:
    twfe_formula += ' + treated:post:risk_score'

basic_coef = np.nan
basic_p = np.nan
basic_ci_low = np.nan
basic_ci_high = np.nan
basic_hetero_coef = np.nan
basic_hetero_p = np.nan
basic_note = 'computed'
basic_view = pd.DataFrame(columns=['term', 'coef', 'p_value'])

if did_ready_for_regression:
    try:
        basic_model = smf.ols(basic_formula, data=did).fit(cov_type='HC3')
        if 'treated:post' in basic_model.params.index:
            basic_coef = float(basic_model.params['treated:post'])
            basic_p = float(basic_model.pvalues['treated:post'])
            ci = basic_model.conf_int().loc['treated:post']
            basic_ci_low, basic_ci_high = float(ci[0]), float(ci[1])
        if 'treated:post:risk_score' in basic_model.params.index:
            basic_hetero_coef = float(basic_model.params['treated:post:risk_score'])
            basic_hetero_p = float(basic_model.pvalues['treated:post:risk_score'])
        basic_view = pd.DataFrame({'term': basic_model.params.index, 'coef': basic_model.params.values, 'p_value': basic_model.pvalues.values}).sort_values('p_value')
    except Exception as exc:
        basic_note = f'skipped_due_to_model_instability: {exc}'
else:
    basic_note = 'skipped_due_to_insufficient_data'

twfe_coef = np.nan
twfe_p = np.nan
twfe_ci_low = np.nan
twfe_ci_high = np.nan
twfe_hetero_coef = np.nan
twfe_hetero_p = np.nan
twfe_note = 'computed'
twfe_view = pd.DataFrame(columns=['term', 'coef', 'p_value'])

if did_ready_for_regression:
    try:
        twfe_model = smf.ols(twfe_formula, data=did).fit(cov_type='cluster', cov_kwds={'groups': did['user_id']})
        if 'treated:post' in twfe_model.params.index:
            twfe_coef = float(twfe_model.params['treated:post'])
            twfe_p = float(twfe_model.pvalues['treated:post'])
            ci = twfe_model.conf_int().loc['treated:post']
            twfe_ci_low, twfe_ci_high = float(ci[0]), float(ci[1])
        if 'treated:post:risk_score' in twfe_model.params.index:
            twfe_hetero_coef = float(twfe_model.params['treated:post:risk_score'])
            twfe_hetero_p = float(twfe_model.pvalues['treated:post:risk_score'])
        twfe_view = pd.DataFrame({'term': twfe_model.params.index, 'coef': twfe_model.params.values, 'p_value': twfe_model.pvalues.values}).sort_values('p_value')
    except Exception as exc:
        twfe_note = f'skipped_due_to_model_instability: {exc}'
else:
    twfe_note = 'skipped_due_to_insufficient_data'

parallel_coef = np.nan
parallel_p = np.nan
parallel_note = 'not_tested'
if did_ready_for_regression:
    pre = did[did['post'] == 0].copy()
    if pre['period_str'].nunique() >= 3 and pre['treated'].nunique() >= 2 and pre['user_id'].nunique() >= MIN_DID_UNITS:
        order = {k: i for i, k in enumerate(sorted(pre['period_str'].unique().tolist()))}
        pre['t_num'] = pre['period_str'].map(order).astype(float)
        try:
            pre_model = smf.ols('outcome ~ treated + t_num + treated:t_num', data=pre).fit(cov_type='HC3')
            parallel_coef = float(pre_model.params.get('treated:t_num', np.nan))
            parallel_p = float(pre_model.pvalues.get('treated:t_num', np.nan))
            parallel_note = 'computed'
        except Exception as exc:
            parallel_note = f'skipped_due_to_model_instability: {exc}'
    else:
        parallel_note = 'skipped_due_to_insufficient_pre_period_support'

# Event-study coefficients.
event_study_table = pd.DataFrame(columns=['rel_period', 'coef', 'p_value'])
event_study_note = 'computed'
if event_study_ready:
    try:
        es_formula = 'outcome ~ C(user_id) + C(period_str) + C(rel_period_cat, Treatment(reference="-1")):treated'
        es_model = smf.ols(es_formula, data=did).fit(cov_type='cluster', cov_kwds={'groups': did['user_id']})
        rows = []
        for term, coef_val in es_model.params.items():
            if 'rel_period_cat' in term and 'treated' in term:
                match = re.search(r'\[T\.(-?\d+)\]', term)
                if match is None:
                    continue
                rel = int(match.group(1))
                pval_term = float(es_model.pvalues.get(term, np.nan))
                rows.append({'rel_period': rel, 'coef': float(coef_val), 'p_value': pval_term})
        event_study_table = pd.DataFrame(rows, columns=['rel_period', 'coef', 'p_value'])
        if not event_study_table.empty:
            event_study_table = event_study_table.sort_values('rel_period').reset_index(drop=True)
        else:
            event_study_note = 'no_relative_period_coefficients_estimated'
    except Exception as exc:
        event_study_note = f'skipped_due_to_model_instability: {exc}'
else:
    event_study_note = 'skipped_due_to_insufficient_data'

# Placebo DiD on pre-period only.
placebo_coef = np.nan
placebo_p = np.nan
placebo_note = 'computed'
if did_ready_for_regression:
    pre_all = did[did['post'] == 0].copy()
    if pre_all['period_idx'].nunique() >= 3 and pre_all['treated'].nunique() >= 2:
        cut = int(np.median(sorted(pre_all['period_idx'].unique().tolist())))
        pre_all['placebo_post'] = (pre_all['period_idx'] >= cut).astype(int)
        if pre_all['placebo_post'].nunique() >= 2:
            placebo_formula = f'outcome ~ treated + placebo_post + treated:placebo_post{cov_terms}'
            try:
                placebo_model = smf.ols(placebo_formula, data=pre_all).fit(cov_type='HC3')
                placebo_coef = float(placebo_model.params.get('treated:placebo_post', np.nan))
                placebo_p = float(placebo_model.pvalues.get('treated:placebo_post', np.nan))
            except Exception as exc:
                placebo_note = f'skipped_due_to_model_instability: {exc}'
        else:
            placebo_note = 'skipped_due_to_single_placebo_period_group'
    else:
        placebo_note = 'skipped_due_to_insufficient_pre_period_support'
else:
    placebo_note = 'skipped_due_to_insufficient_data'

if not basic_view.empty:
    print('Basic DiD coefficients:')
    display(basic_view)
else:
    print('Basic DiD unavailable:', basic_note)

if not twfe_view.empty:
    print('TWFE DiD coefficients:')
    display(twfe_view.head(20))
else:
    print('TWFE DiD unavailable:', twfe_note)

if not event_study_table.empty:
    display(event_study_table)
else:
    print('Event-study unavailable:', event_study_note)

print('Basic DiD treated:post coef/p:', basic_coef, basic_p, '| CI:', (basic_ci_low, basic_ci_high), '| note:', basic_note)
print('TWFE DiD treated:post coef/p:', twfe_coef, twfe_p, '| CI:', (twfe_ci_low, twfe_ci_high), '| note:', twfe_note)
print('Parallel trends p:', parallel_p, '| note:', parallel_note)
print('Placebo DiD p:', placebo_p, '| note:', placebo_note)





Basic DiD coefficients:


,term,coef,p_value
4,pre_orders,0.790248,2.041368e-124
5,risk_score,-1.725060,3.582056e-17
0,Intercept,1.320316,1.081002e-15
3,treated:post,1.204073,1.064967e-05
6,treated:post:risk_score,-1.281301,2.341011e-04
2,post,0.201159,3.915535e-02
1,treated,-0.044549,6.046107e-01


TWFE DiD coefficients:


,term,coef,p_value
56,C(user_id)[T.81],0.152900,0.000000e+00
26,C(user_id)[T.34],0.050200,0.000000e+00
83,C(user_id)[T.122],0.045300,0.000000e+00
45,C(user_id)[T.64],-0.089500,0.000000e+00
20,C(user_id)[T.28],-0.325900,0.000000e+00
130,C(user_id)[T.192],2.986346,0.000000e+00
63,C(user_id)[T.94],0.075400,0.000000e+00
31,C(user_id)[T.43],-0.045200,0.000000e+00
58,C(user_id)[T.86],-0.069100,0.000000e+00
54,C(user_id)[T.79],0.057200,0.000000e+00


Event-study unavailable: no_relative_period_coefficients_estimated
Basic DiD treated:post coef/p: 1.2040727396147743 1.0649671847605551e-05 | CI: (0.668154566685829, 1.7399909125437198) | note: computed
TWFE DiD treated:post coef/p: 1.62696070722666 2.40226026549218e-07 | CI: (1.0095972805676674, 2.2443241338856526) | note: computed
Parallel trends p: 0.25412001702228415 | note: computed
Placebo DiD p: 0.2233936524259027 | note: computed


In [9]:
# Cell 8: Save DiD report + decision matrix with reason codes

def _num_or_none(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return float(x)

# Use TWFE as primary DiD estimate when available, fallback to basic.
primary_did_coef = twfe_coef if pd.notna(twfe_coef) else basic_coef
primary_did_p = twfe_p if pd.notna(twfe_p) else basic_p
primary_did_model = 'twfe_clustered' if pd.notna(twfe_coef) else 'basic_hc3'

ab_binary_stat_ok = ab_binary_ready and pd.notna(p_value) and float(p_value) < ALPHA
ab_binary_practical_ok = pd.notna(rate_diff) and float(rate_diff) >= PRACTICAL_MIN_CONVERSION_LIFT
ab_revenue_stat_ok = ab_continuous_ready and pd.notna(cuped_p) and float(cuped_p) < ALPHA
ab_revenue_practical_ok = pd.notna(cuped_diff) and float(cuped_diff) >= PRACTICAL_MIN_REVENUE_LIFT

did_stat_ok = did_ready_for_regression and pd.notna(primary_did_p) and float(primary_did_p) < ALPHA
did_practical_ok = pd.notna(primary_did_coef) and float(primary_did_coef) >= PRACTICAL_MIN_DID_EFFECT
pretrend_ok = (pd.isna(parallel_p) and parallel_note != 'computed') or (pd.notna(parallel_p) and float(parallel_p) >= ALPHA)
placebo_ok = (pd.isna(placebo_p) and placebo_note != 'computed') or (pd.notna(placebo_p) and float(placebo_p) >= ALPHA)

reason_codes = []
if srm_flag:
    reason_codes.append('SRM_FAIL')
if not ab_binary_ready:
    reason_codes.append('AB_BINARY_NOT_READY')
if not ab_continuous_ready:
    reason_codes.append('AB_CONT_NOT_READY')
if not did_ready_for_regression:
    reason_codes.append('DID_NOT_READY')
if not ab_binary_stat_ok and ab_binary_ready:
    reason_codes.append('AB_BINARY_NOT_SIGNIFICANT')
if ab_binary_stat_ok and not ab_binary_practical_ok:
    reason_codes.append('AB_BINARY_NOT_PRACTICAL')
if not ab_revenue_stat_ok and ab_continuous_ready:
    reason_codes.append('AB_REVENUE_NOT_SIGNIFICANT')
if ab_revenue_stat_ok and not ab_revenue_practical_ok:
    reason_codes.append('AB_REVENUE_NOT_PRACTICAL')
if not did_stat_ok and did_ready_for_regression:
    reason_codes.append('DID_NOT_SIGNIFICANT')
if did_stat_ok and not did_practical_ok:
    reason_codes.append('DID_NOT_PRACTICAL')
if not pretrend_ok:
    reason_codes.append('PARALLEL_TRENDS_FAIL')
if not placebo_ok:
    reason_codes.append('PLACEBO_FAIL')

if not ab_binary_ready and not ab_continuous_ready and not did_ready_for_regression:
    final_decision = 'collect_data'
elif srm_flag:
    final_decision = 'iterate'
elif ab_binary_stat_ok and ab_binary_practical_ok and did_stat_ok and did_practical_ok and pretrend_ok and placebo_ok:
    final_decision = 'ship'
elif (ab_binary_stat_ok and ab_binary_practical_ok) or (ab_revenue_stat_ok and ab_revenue_practical_ok) or (did_stat_ok and did_practical_ok):
    final_decision = 'pilot'
else:
    final_decision = 'iterate'

if not reason_codes:
    reason_codes.append('ALL_CHECKS_PASS')

did_results = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'readiness': {
        'did_ready_for_manual': bool(did_ready_for_manual),
        'did_ready_for_regression': bool(did_ready_for_regression),
        'did_skip_reasons': did_skip_reasons,
        'basic_note': basic_note,
        'twfe_note': twfe_note,
        'event_study_note': event_study_note,
        'parallel_note': parallel_note,
        'placebo_note': placebo_note,
    },
    'manual': {k: _num_or_none(v) for k, v in manual_overall.items()},
    'manual_by_risk_band': risk_did_table.to_dict(orient='records'),
    'basic_regression': {
        'formula': basic_formula,
        'coef_treated_post': _num_or_none(basic_coef),
        'coef_ci_low': _num_or_none(basic_ci_low),
        'coef_ci_high': _num_or_none(basic_ci_high),
        'p_value': _num_or_none(basic_p),
        'coef_treated_post_risk_score': _num_or_none(basic_hetero_coef),
        'p_value_treated_post_risk_score': _num_or_none(basic_hetero_p),
    },
    'twfe_regression': {
        'formula': twfe_formula,
        'coef_treated_post': _num_or_none(twfe_coef),
        'coef_ci_low': _num_or_none(twfe_ci_low),
        'coef_ci_high': _num_or_none(twfe_ci_high),
        'p_value': _num_or_none(twfe_p),
        'coef_treated_post_risk_score': _num_or_none(twfe_hetero_coef),
        'p_value_treated_post_risk_score': _num_or_none(twfe_hetero_p),
    },
    'parallel_trends_test': {
        'coef_treated_time_pre': _num_or_none(parallel_coef),
        'p_value': _num_or_none(parallel_p),
    },
    'event_study': event_study_table.to_dict(orient='records'),
    'placebo_did': {
        'coef_treated_placebo_post': _num_or_none(placebo_coef),
        'p_value': _num_or_none(placebo_p),
    },
}

did_out = DID_DIR / 'did_results.json'
did_out.write_text(json.dumps(did_results, indent=2, default=str), encoding='utf-8')

if isinstance(event_study_table, pd.DataFrame):
    event_study_table.to_csv(DID_DIR / 'did_event_study_coefficients.csv', index=False)


decision = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'final_decision': final_decision,
    'reason_codes': reason_codes,
    'alpha': ALPHA,
    'thresholds': {
        'practical_min_conversion_lift': PRACTICAL_MIN_CONVERSION_LIFT,
        'practical_min_revenue_lift': PRACTICAL_MIN_REVENUE_LIFT,
        'practical_min_did_effect': PRACTICAL_MIN_DID_EFFECT,
    },
    'checks': {
        'srm_flag': bool(srm_flag),
        'ab_binary_ready': bool(ab_binary_ready),
        'ab_continuous_ready': bool(ab_continuous_ready),
        'did_ready_for_regression': bool(did_ready_for_regression),
        'ab_binary_stat_ok': bool(ab_binary_stat_ok),
        'ab_binary_practical_ok': bool(ab_binary_practical_ok),
        'ab_revenue_stat_ok': bool(ab_revenue_stat_ok),
        'ab_revenue_practical_ok': bool(ab_revenue_practical_ok),
        'did_stat_ok': bool(did_stat_ok),
        'did_practical_ok': bool(did_practical_ok),
        'pretrend_ok': bool(pretrend_ok),
        'placebo_ok': bool(placebo_ok),
        'primary_did_model': primary_did_model,
        'primary_did_coef': _num_or_none(primary_did_coef),
        'primary_did_p': _num_or_none(primary_did_p),
    },
    'skip_reasons': {
        'ab_binary': ab_binary_skip_reasons,
        'ab_heterogeneity': ab_heterogeneity_skip_reasons,
        'did': did_skip_reasons,
    },
}

decision_out = CAUSAL_DIR / 'causal_release_decision.json'
decision_out.write_text(json.dumps(decision, indent=2), encoding='utf-8')

print('Saved DiD results:', did_out)
print('Saved causal decision:', decision_out)
print('Final decision:', final_decision)
print('Reason codes:', reason_codes)



Saved DiD results: /Users/deliorincon/Desktop/Sliceiq/ml/data/reports/causal/did/did_results.json
Saved causal decision: /Users/deliorincon/Desktop/Sliceiq/ml/data/reports/causal/causal_release_decision.json
Final decision: ship
Reason codes: ['ALL_CHECKS_PASS']


## Next Notebook

Proceed to `05_cohort_time_series.ipynb`.
